# CCA-F Mock Exam — Question Generator

Generates a fresh set of 25 exam-grade questions for the **Claude Certified Architect – Foundations (CCA-F)** exam,  
then exports them to a structured `.md` file with solutions and anti-pattern analysis.

---
## Setup
```bash
pip install anthropic
```

In [1]:
# ── 1. CONFIGURATION ─────────────────────────────────────────────────────────
import os
from datetime import datetime

# Set your Anthropic API key here or export as an environment variable:
# export ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE")

# ── Model selection ───────────────────────────────────────────────────────────
# Haiku  → fast, cheap,        good for iteration / testing prompts
# Sonnet → balanced quality,   recommended for realistic exam questions  ← default
# Opus   → highest quality,    slowest and most expensive
MODEL = "claude-sonnet-4-20250514"
# MODEL = "claude-haiku-4-5-20251001"
# MODEL = "claude-opus-4-20250514"

# ── Token budget per batch ────────────────────────────────────────────────────
# 3 questions × ~1000-1200 tokens each (full-sentence options) = ~3600 tokens
# 4000 gives comfortable headroom for Sonnet's richer output
MAX_TOKENS = 4000

# ── Output file ───────────────────────────────────────────────────────────────
OUTPUT_FILE = f"qna_{datetime.now().strftime('%Y-%m-%dT%H-%M-%S')}.md"

print(f"Model     : {MODEL}")
print(f"Max tokens: {MAX_TOKENS}")
print(f"Output    : {OUTPUT_FILE}")

Model     : claude-sonnet-4-20250514
Max tokens: 4000
Output    : qna_2026-06-02T21-06-48.md


In [2]:
# ── 2. DOMAIN DEFINITIONS & BATCH LAYOUT ─────────────────────────────────────

DOMAIN_NAMES = {
    1: "Agentic Architecture & Orchestration",
    2: "Tool Design & MCP Integration",
    3: "Claude Code Configuration & Workflows",
    4: "Prompt Engineering & Structured Output",
    5: "Context Management & Reliability",
}

DOMAIN_WEIGHTS = {1: "27%", 2: "18%", 3: "20%", 4: "20%", 5: "15%"}

# 9 batches: 8 × 3 questions + 1 × 1 question = 25 total
# Domain distribution achieved: D1=7, D2=5, D3=5, D4=5, D5=3
BATCHES = [
    [(1,1),(2,2),(3,3)],       # Batch 1 — D1, D2, D3
    [(4,1),(5,4),(6,5)],       # Batch 2 — D1, D4, D5
    [(7,1),(8,2),(9,3)],       # Batch 3 — D1, D2, D3
    [(10,4),(11,5),(12,1)],    # Batch 4 — D4, D5, D1
    [(13,2),(14,3),(15,4)],    # Batch 5 — D2, D3, D4
    [(16,1),(17,2),(18,3)],    # Batch 6 — D1, D2, D3
    [(19,4),(20,5),(21,1)],    # Batch 7 — D4, D5, D1
    [(22,2),(23,4),(24,3)],    # Batch 8 — D2, D4, D3
    [(25,1)],                  # Batch 9 — D1 (single question)
]

from collections import Counter
dist = Counter(d for batch in BATCHES for _, d in batch)
print(f"Total questions : {sum(dist.values())}")
print(f"Total batches   : {len(BATCHES)} (8 × 3q + 1 × 1q)")
print()
print("Domain distribution:")
for d, name in DOMAIN_NAMES.items():
    bar = "█" * dist[d]
    print(f"  D{d} ({DOMAIN_WEIGHTS[d]}) {bar} {dist[d]}q — {name}")

Total questions : 25
Total batches   : 9 (8 × 3q + 1 × 1q)

Domain distribution:
  D1 (27%) ███████ 7q — Agentic Architecture & Orchestration
  D2 (18%) █████ 5q — Tool Design & MCP Integration
  D3 (20%) █████ 5q — Claude Code Configuration & Workflows
  D4 (20%) █████ 5q — Prompt Engineering & Structured Output
  D5 (15%) ███ 3q — Context Management & Reliability


In [3]:
# ── 3. PROMPT BUILDER ────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "Return ONLY a raw JSON array. "
    "No markdown, no code fences, no explanation. "
    "Start with [ and end with ]."
)


def make_prompt(items: list[tuple[int, int]]) -> str:
    """
    Build the user prompt for a batch of questions.

    Args:
        items: list of (question_id, domain) tuples

    Returns:
        Formatted prompt string ready to send to Claude
    """
    lines = "\n".join(
        f"ID {qid}: Domain {dom} ({DOMAIN_NAMES[dom]})"
        for qid, dom in items
    )
    ids = [qid for qid, _ in items]

    return f"""Write exactly {len(items)} CCA-F certification exam questions about building Claude-based AI systems.

Questions:
{lines}

Requirements:
- Generic SaaS/e-commerce/devtools/enterprise scenarios only.
- 4 options each, exactly one correct answer.
- Each option must be 1-2 complete sentences describing a full architectural approach.
- Wrong options must be plausible mistakes, not obviously wrong.
- Test architectural judgment, not API memorisation.
- Scenarios to use: Customer Support Agent, E-commerce Platform, Multi-Agent Pipeline,
  Developer Productivity Tool, Code Review CI/CD, Content Moderation, Data Extraction,
  Enterprise Knowledge Base, Marketing Automation.
- Every question must be distinct — no repeated concepts across the batch.
- Difficulty: medium to hard, scenario-based.
- "explanation": 1-2 complete sentences why the correct answer is right.
- "antipattern_index": index (0-3) of the most dangerous wrong option (must differ from "answer").
- "antipattern_reason": 1 sentence why it is an anti-pattern. Use Claude terminology where
  applicable: context bloat, non-idempotent tool, missing circuit breaker, unbounded retry,
  prompt injection surface, context saturation, raw object parameter.

Output a JSON array of exactly {len(items)} objects with IDs {ids}:
[{{"id":N,"domain":N,"scenario":"name","question":"text","options":["A","B","C","D"],
  "answer":N,"explanation":"text","antipattern_index":N,"antipattern_reason":"text"}}]"""


# Preview the prompt for batch 1
print(make_prompt(BATCHES[0]))

Write exactly 3 CCA-F certification exam questions about building Claude-based AI systems.

Questions:
ID 1: Domain 1 (Agentic Architecture & Orchestration)
ID 2: Domain 2 (Tool Design & MCP Integration)
ID 3: Domain 3 (Claude Code Configuration & Workflows)

Requirements:
- Generic SaaS/e-commerce/devtools/enterprise scenarios only.
- 4 options each, exactly one correct answer.
- Each option must be 1-2 complete sentences describing a full architectural approach.
- Wrong options must be plausible mistakes, not obviously wrong.
- Test architectural judgment, not API memorisation.
- Scenarios to use: Customer Support Agent, E-commerce Platform, Multi-Agent Pipeline,
  Developer Productivity Tool, Code Review CI/CD, Content Moderation, Data Extraction,
  Enterprise Knowledge Base, Marketing Automation.
- Every question must be distinct — no repeated concepts across the batch.
- Difficulty: medium to hard, scenario-based.
- "explanation": 1-2 complete sentences why the correct answer is r

In [4]:
# ── 4. QUESTION GENERATION ───────────────────────────────────────────────────
import anthropic
import json
import time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def call_batch(items: list[tuple[int, int]], retries: int = 2) -> list[dict]:
    """
    Send one batch to Claude and return parsed questions.
    Retries on JSON parse failure up to `retries` times.
    """
    for attempt in range(retries + 1):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": make_prompt(items)}],
            )
            raw = response.content[0].text.strip()
            # Strip any accidental markdown fences
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            if attempt < retries:
                print(f"    JSON parse failed (attempt {attempt+1}/{retries+1}), retrying... ({e})")
                time.sleep(2)
            else:
                raise RuntimeError(f"Batch failed after {retries+1} attempts: {e}")


def generate_all_questions() -> list[dict]:
    """
    Run all 9 batches sequentially.
    Progress is printed after each batch — if a batch fails, already-generated
    questions are preserved in `all_questions` so you can inspect or resume.
    """
    all_questions = []

    for i, batch in enumerate(BATCHES):
        id_range = f"ID {batch[0][0]}" if len(batch) == 1 else f"IDs {batch[0][0]}–{batch[-1][0]}"
        print(
            f"Batch {i+1}/{len(BATCHES)} — {len(batch)}q ({id_range})...",
            end=" ", flush=True
        )
        try:
            questions = call_batch(batch)
            all_questions.extend(questions)
            print(f"✓  ({len(all_questions)} total saved)")
        except Exception as e:
            print(f"\n✗ FAILED: {e}")
            print(f"  {len(all_questions)} questions saved from {i} completed batch(es).")
            print("  Re-run this cell to retry from scratch, or inspect `all_questions` below.")
            break

    # Re-index IDs 1–N in order received
    for idx, q in enumerate(all_questions):
        q["id"] = idx + 1

    return all_questions


questions = generate_all_questions()
print(f"\n{'='*50}")
print(f"Generation complete: {len(questions)} / 25 questions")

Batch 1/9 — 3q (IDs 1–3)... ✓  (3 total saved)
Batch 2/9 — 3q (IDs 4–6)... ✓  (6 total saved)
Batch 3/9 — 3q (IDs 7–9)... ✓  (9 total saved)
Batch 4/9 — 3q (IDs 10–12)... ✓  (12 total saved)
Batch 5/9 — 3q (IDs 13–15)... ✓  (15 total saved)
Batch 6/9 — 3q (IDs 16–18)... ✓  (18 total saved)
Batch 7/9 — 3q (IDs 19–21)... ✓  (21 total saved)
Batch 8/9 — 3q (IDs 22–24)... ✓  (24 total saved)
Batch 9/9 — 1q (ID 25)... ✓  (25 total saved)

Generation complete: 25 / 25 questions


In [5]:
# ── 5. PREVIEW SAMPLE QUESTIONS ──────────────────────────────────────────────

LABELS = ["A", "B", "C", "D"]


def preview_question(q: dict) -> None:
    print(f"Q{q['id']}. [{DOMAIN_NAMES[q['domain']]}] — {q['scenario']}")
    print(f"   {q['question']}")
    print()
    for i, opt in enumerate(q["options"]):
        if i == q["answer"]:
            marker = "✓"
        elif i == q.get("antipattern_index"):
            marker = "⚠"
        else:
            marker = " "
        print(f"   {marker} {LABELS[i]}. {opt}")
    print()
    print(f"   Explanation  : {q['explanation']}")
    print(f"   Anti-pattern : ({LABELS[q.get('antipattern_index', 0)]}) {q.get('antipattern_reason', '')}")
    print("-" * 80)


# Show first 3 questions as a quality sanity check
for q in questions[:3]:
    preview_question(q)

Q1. [Agentic Architecture & Orchestration] — Multi-Agent Pipeline
   You're designing a multi-agent pipeline for content moderation where Agent A classifies content type, Agent B performs safety analysis, and Agent C generates moderation decisions. How should you architect the orchestration layer to handle agent failures and maintain system reliability?

   ✓ A. Implement a centralized orchestrator with circuit breakers for each agent, exponential backoff on retries, and fallback workflows that can complete moderation with reduced agent participation.
   ⚠ B. Chain the agents sequentially with direct API calls between them, implementing retry logic in each agent to handle downstream failures and maintaining shared state in a database.
     C. Use a pub-sub messaging system where each agent subscribes to the previous agent's output topic, with dead letter queues for failed messages and manual intervention processes.
     D. Deploy agents as microservices with load balancers and let each

In [6]:
# ── 6. DOMAIN DISTRIBUTION CHECK ─────────────────────────────────────────────

dist = Counter(q["domain"] for q in questions)
print("Domain distribution in generated set:")
print(f"{'Domain':<6} {'Weight':<8} {'Count':<6} {'Bar':<10} Name")
print("-" * 60)
for d, name in DOMAIN_NAMES.items():
    count = dist.get(d, 0)
    bar   = "█" * count
    print(f"  D{d}    {DOMAIN_WEIGHTS[d]:<8} {count:<6} {bar:<10} {name}")

Domain distribution in generated set:
Domain Weight   Count  Bar        Name
------------------------------------------------------------
  D1    27%      7      ███████    Agentic Architecture & Orchestration
  D2    18%      5      █████      Tool Design & MCP Integration
  D3    20%      5      █████      Claude Code Configuration & Workflows
  D4    20%      5      █████      Prompt Engineering & Structured Output
  D5    15%      3      ███        Context Management & Reliability


In [7]:
# ── 7. MARKDOWN EXPORT ───────────────────────────────────────────────────────


def build_markdown(questions: list[dict]) -> str:
    ts = datetime.now().strftime("%d %b %Y, %I:%M %p")

    # ── Part 1: Questions only (no answers visible) ───────────────────────────
    q_sections = []
    for i, q in enumerate(questions):
        opts = "\n".join(
            f"  - **{LABELS[oi]}.** {opt}"
            for oi, opt in enumerate(q["options"])
        )
        q_sections.append(
            f"### Q{i+1}. [Domain {q['domain']} — {DOMAIN_NAMES[q['domain']]}]\n"
            f"**Scenario:** {q['scenario']}\n\n"
            f"{q['question']}\n\n"
            f"{opts}"
        )

    # ── Part 2: Solutions with anti-pattern analysis ──────────────────────────
    s_sections = []
    for i, q in enumerate(questions):
        preview   = q["question"][:75] + ("…" if len(q["question"]) > 75 else "")
        ap_idx    = q.get("antipattern_index", 0)
        ap_label  = LABELS[ap_idx]
        ap_option = q["options"][ap_idx]
        ap_reason = q.get("antipattern_reason", "_Not available_")
        s_sections.append(
            f"### Q{i+1}. {preview}\n\n"
            f"**✅ Correct Answer: {LABELS[q['answer']]}**\n"
            f"> {q['options'][q['answer']]}\n\n"
            f"{q['explanation']}\n\n"
            f"**⚠️ Anti-Pattern: {ap_label}**\n"
            f"> {ap_option}\n\n"
            f"{ap_reason}"
        )

    return (
        f"# CCA-F Mock Exam — Questions & Solutions\n"
        f"_Generated: {ts} · Model: {MODEL}_\n\n"
        f"---\n\n"
        f"## Part 1 — Questions\n\n"
        + "\n\n---\n\n".join(q_sections)
        + "\n\n---\n\n"
        + "## Part 2 — Solutions\n\n"
        + "\n\n---\n\n".join(s_sections)
        + "\n"
    )


md_content = build_markdown(questions)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"✓ Saved {len(questions)} questions to: {OUTPUT_FILE}")
print(f"  File size: {len(md_content):,} characters")

✓ Saved 25 questions to: qna_2026-06-02T21-06-48.md
  File size: 51,283 characters


In [8]:
# ── 8. DOWNLOAD ──────────────────────────────────────────────────────────────
# Auto-detects Colab vs local Jupyter.

if os.path.exists("/content"):  # Google Colab
    from google.colab import files
    files.download(OUTPUT_FILE)
    print(f"Downloading {OUTPUT_FILE} via Colab...")
else:  # Local Jupyter / VS Code
    abs_path = os.path.abspath(OUTPUT_FILE)
    print(f"File saved locally at:\n  {abs_path}")
    print("Open in any Markdown viewer, Obsidian, or VS Code Preview.")

File saved locally at:
  /Users/rajeevkulkarni/ml-explorations/cca-exam/notebooks/qna_2026-06-02T21-06-48.md
Open in any Markdown viewer, Obsidian, or VS Code Preview.


In [9]:
# ── 9. (OPTIONAL) SAVE RAW JSON ──────────────────────────────────────────────
# Useful for reloading questions into the web app or running further analysis.

json_file = OUTPUT_FILE.replace(".md", ".json")
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)

print(f"✓ Raw JSON saved to: {json_file}")

✓ Raw JSON saved to: qna_2026-06-02T21-06-48.json
